[![Colab Badge Link](https://img.shields.io/badge/open-in%20colab-blue)](https://colab.research.google.com/github/Glasgow-AI4BioMed/tutorials/blob/main/sentence_level_relation_extraction_with_BERT.ipynb)

# Sentence-level Relation Extraction with BERT

This notebook illustrates how to turn a relation extraction problem into a text classification problem and train a BERT classifier for it.

It uses the [DDICorpus](https://github.com/isegura/DDICorpus) which contains text annotated for drug-drug interactions. We're going to find every candidate relation, add tags around the associated entities (as shown below) and train a text classifier to predict the relation label.

```
It was shown that [ARG1]Warfarin[/ARG1] interacted with [ARG2]aspirin[/ARG2]. -> INTERACTS
We studied [ARG1]Amoxicillin[/ARG1], Doxycycline and [ARG2]Clarithromycin[/ARG2]. -> NONE
```

## Install prerequisites

We're going to use the [bioc](https://github.com/bionlplab/bioc) package for loading the dataset.

In [ ]:
import sys
print(sys.executable)

In [ ]:
!which python

In [ ]:
!pip install bioc

In [ ]:
!pip install spacy

## Get the dataset

Let's download the dataset in [BRAT format](https://brat.nlplab.org/standoff.html):

In [ ]:
!wget -O DDICorpus-2013_BRAT.zip https://github.com/isegura/DDICorpus/raw/refs/heads/master/DDICorpus-2013\(BRAT\).zip
!python -m zipfile -e DDICorpus-2013_BRAT.zip .

We can see that there are a lot of *.txt and *.ann files

In [ ]:
!find DDICorpusBrat/ | head -n 20

Each document is represented by a .txt file that contains the document text, and a .ann file that contains annotations of entities and relations. Let's examine one:

In [ ]:
with open('DDICorpusBrat/Train/DrugBank/Abatacept_ddi.txt') as f:
  text = f.read()

with open('DDICorpusBrat/Train/DrugBank/Abatacept_ddi.ann') as f:
  annotations = f.read()

print(text)

print(annotations)

The text is a long string containing all the sentences in the document. The annotations have the entities (with IDs, coordinates and types) and the relations (which reference the entities using the IDs).

## Load a single document

We'll use the bioc library to load a single file and see what's in there

In [ ]:

from bioc import brat

with open('DDICorpusBrat/Train/DrugBank/Abatacept_ddi.ann') as ann_fp, open('DDICorpusBrat/Train/DrugBank/Abatacept_ddi.txt') as text_fp:
    doc = brat.load(text_fp, ann_fp)


We can see the text of the document:

In [ ]:
doc.text

The entities with associated info:

In [ ]:
for e in doc.entities:
  print(f"{e.id}\t{e.type}\t{e.locations.begin()}\t{e.locations.end()}\t{e.text}")

And the relations:

In [ ]:
for rel in doc.relations:
  print(rel.type, rel.arguments)

## Load the files

Let's load all of the documents for the training set:

In [ ]:
import os

train_docs = []
for directory in ['DrugBank', 'MedLine']:
  for filename in os.listdir(f'DDICorpusBrat/Train/{directory}/'):
    if filename.endswith('.txt'):
      ann_filename = filename.replace('.txt', '.ann')
      with open(f'DDICorpusBrat/Train/{directory}/{ann_filename}') as ann_fp, open(f'DDICorpusBrat/Train/{directory}/{filename}') as text_fp:
        doc = brat.load(text_fp, ann_fp)
        train_docs.append(doc)
len(train_docs)

We'll actually split the training documents into a training and validation set.

In [ ]:
from sklearn.model_selection import train_test_split

train_docs, val_docs = train_test_split(train_docs, test_size=0.2, random_state=42)

len(train_docs), len(val_docs)

There is a test set in the DDI corpus, but we won't use this here. It's a good idea to ignore the test set until final experiments.

## Make things sentence-level

The loaded documents contain multiple sentences. Let's simplify things down to one sentence, one document. This should guarantee that the document text will fit in the context window of a BERT model, and also limit the number of possible relations that need to be considered.

We'll use [spaCy](https://spacy.io) to split the text into sentences and figure out which entities and relations are contained within each sentence. Let's load up the spaCy English model.

In [ ]:
import sys
!{sys.executable} -m spacy download en_core_web_sm

import spacy
nlp = spacy.load("en_core_web_sm")

And now a function that converts a document into a series of documents (one per sentence) with entities and relations contained in each:

In [ ]:
from bioc.brat.datastructure import BratDocument

def make_sentence_level(doc):
  parsed = nlp(doc.text)

  sent_docs = []
  for sent in parsed.sents:
    # Find the entities that are wholly within this sentence
    sent_entities = [ e for e in doc.entities if sent.start_char <= e.locations.begin() and e.locations.end() <= sent.end_char ]

    # Shift the entity locations so they're offsets from the sentence start
    sent_entities = [ e.shift(-sent.start_char) for e in sent_entities ]

    # Find relations whose arguments are wholly within this sentence
    sent_entity_ids = { e.id for e in sent_entities }
    sent_relations = [ rel for rel in doc.relations if all(e_id in sent_entity_ids for e_id in rel.arguments.values()) ]

    # Create a new document with just this sentence and associated entities/relations
    sent_doc = BratDocument()
    sent_doc.text = sent.text
    sent_doc.annotations += sent_entities
    sent_doc.annotations += sent_relations

    sent_docs.append(sent_doc)

  return sent_docs

Apply that function to the loaded training and test documents

In [ ]:
from tqdm.auto import tqdm

train_sent_docs = []
for doc in tqdm(train_docs):
  train_sent_docs += make_sentence_level(doc)

val_sent_docs = []
for doc in tqdm(val_docs):
  val_sent_docs += make_sentence_level(doc)

The number of documents will increase dramatically (as we now have one document per sentence):

In [ ]:
len(train_docs), len(train_sent_docs)

However, the number of relations actually decreases:

In [ ]:
sum( len(doc.relations) for doc in train_docs ), sum( len(doc.relations) for doc in train_sent_docs )

This can be for a number of reasons:
- Entities that cross sentence boundaries are excluded (and hence associated relations are removed)
- Relations that cross sentence boundaries are excluded

**Note:** If the task was to do relation extraction on these documents, then these missing relations would count as false negatives (and should be properly factored in reported performance metrics). We won't do that here, but something to bear in mind.

### Examine a single sentence

Let's see an example of a single sentence document and associated entity and relation annotations:

In [ ]:
doc = train_sent_docs[0]
doc.text

In [ ]:
for e in doc.entities:
  print(f"{e.id}\t{e.type}\t{e.locations.begin()}\t{e.locations.end()}\t{e.text}")

In [ ]:
for rel in doc.relations:
  print(rel.type, rel.arguments)

## Turning relation extraction into text classification

Now let's do a quick check. All the relations have two arguments called 'Arg1' and 'Arg2':

In [ ]:
for doc in train_sent_docs:
  for rel in doc.relations:
    assert sorted(rel.arguments.keys()) == ['Arg1','Arg2']

We want to find all the possible candidate relations in a document (which is every possible pair of entities). For each candidate pair, we need to know if it has a labelled relationship (or otherwise it's negative). Let's go through the steps for a single doc:

In [ ]:
doc = train_sent_docs[0]

First let's make a lookup for the labelled relations based on the entities for Arg1/Arg2:

In [ ]:
candidate_to_label = {}
for rel in doc.relations:
  candidate_to_label[(rel.arguments['Arg1'], rel.arguments['Arg2'])] = rel.type
candidate_to_label

Now we iterate through every possible candidate relation (i.e. pair of entities), make a version of the text where the entities are tagged, and figure out their label (which may be NONE).

In [ ]:
import itertools

for e1,e2 in itertools.product(doc.entities, doc.entities):
  if e1.id != e2.id:
    # Figure out where we want to insert the entity open/close tags
    inserts = [ (e1.locations.begin(), '[E1]'), (e1.locations.end(), '[/E1]'), (e2.locations.begin(), '[E2]'), (e2.locations.end(), '[/E2]') ]

    # Sort the insertions in reverse order (so they don't affect the text as they are done)
    inserts = sorted(inserts, key=lambda x: x[0], reverse=True)

    # Iterate through the insertion and put the tags into the text
    new_text = doc.text
    for insert in inserts:
      new_text = new_text[:insert[0]] + insert[1] + new_text[insert[0]:]

    # Get the label and default to NONE
    label = candidate_to_label.get((e1.id, e2.id), 'NONE')

    print(f"{label}\t{new_text}")


### Create text classification datasets

Let's turn that into a function:

In [ ]:
def make_labelled_data(doc):
  candidate_to_label = {}
  for rel in doc.relations:
    candidate_to_label[(rel.arguments['Arg1'], rel.arguments['Arg2'])] = rel.type

  labelled_data = []
  for e1,e2 in itertools.product(doc.entities, doc.entities):
    if e1.id != e2.id:
      inserts = [ (e1.locations.begin(), '[E1]'), (e1.locations.end(), '[/E1]'), (e2.locations.begin(), '[E2]'), (e2.locations.end(), '[/E2]') ]
      inserts = sorted(inserts, key=lambda x: x[0], reverse=True)

      new_text = doc.text
      for insert in inserts:
        new_text = new_text[:insert[0]] + insert[1] + new_text[insert[0]:]

      label = candidate_to_label.get((e1.id, e2.id), 'NONE')

      labelled_data.append({'text': new_text, 'label': label})

  return labelled_data

And apply it across the whole dataset:

In [ ]:
train_data = sum( [ make_labelled_data(doc) for doc in train_sent_docs ], [] )
val_data = sum( [ make_labelled_data(doc) for doc in val_sent_docs ], [] )

The dataset has now got much larger as we create a sample for every possible candidate relation

In [ ]:
len(train_sent_docs), len(train_data)

If we count up the labels, it is dominated by the negative (NONE) label:

In [ ]:
from collections import Counter
Counter(x['label'] for x in train_data)

### Sample down the training set for speed

For the purpose of this notebook, let's make the training set smaller (as it'll take a long time to run if we don't). We'll keep all the actual relations and reduce the number of negatives (i.e. those labelled as NONE)

In [ ]:
len(train_data)

In [ ]:
positive_data = [ x for x in train_data if x['label'] != 'NONE' ]
negative_data = [ x for x in train_data if x['label'] == 'NONE' ]

import random
random.seed(42)

# Make the data have 5 times the negative examples to non-negative
smaller_train_data = positive_data + random.sample(negative_data,5*len(positive_data))
len(smaller_train_data)

### Train a relation classifier

Now we have the dataset set up for a traditional multi-class text classification problem. Let's turn our dataset into a [DatasetDict](https://huggingface.co/docs/datasets/en/package_reference/main_classes#datasets.DatasetDict)/[Dataset](https://huggingface.co/docs/datasets/en/package_reference/main_classes#datasets.Dataset) so that it plays nicely with the Hugging Face trainer.

In [ ]:
from datasets import Dataset, DatasetDict

dataset = DatasetDict({
    "train": Dataset.from_list(smaller_train_data),
    "validation": Dataset.from_list(val_data)
})

We'll also pull out the label information and label mapping. Plus let's convert the labels to corresponding integer values.

In [ ]:
unique_labels = sorted(set(dataset["train"]["label"]))
label2id = {l: i for i, l in enumerate(unique_labels)}
id2label = {i: l for l, i in label2id.items()}

def encode_labels(example):
    example["label"] = label2id[example["label"]]
    return example

dataset = dataset.map(encode_labels)

We're going to use the [microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext](https://huggingface.co/microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext) model. Let's get its tokenizer.

In [ ]:
from transformers import AutoTokenizer

model_name = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
tokenizer = AutoTokenizer.from_pretrained(model_name)

We'll add tags around the entities (e.g. [Arg1]aspirin[/Arg1]) and it'd make sense for those tags to be single tokens (and not be split into multiple tokens). So let's add them to the tokenizer:

In [ ]:
tokenizer.add_tokens(["[E1]", "[/E1]", "[E2]", "[/E2]"])

And apply the tokenizer to our data:

In [ ]:
def tokenize_fn(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding=False,
        max_length=512
    )

dataset = dataset.map(tokenize_fn, batched=True)

In [ ]:
print(tokenizer.tokenize("[E1]aspirin[/E1] and [Arg1]warfarin[/Arg1]"))

Now we create the text classifier model with the corresponding number of output labels

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(unique_labels),
    id2label=id2label,
    label2id=label2id,
)

# And tell the model to adjust for the extra tokens ([Arg1], etc) we added
model.resize_token_embeddings(len(tokenizer))

Set up the `compute_metrics` function to calculate the typical performance metrics for a multi-class classification problem:

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro"
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "macro_f1": f1,
        "macro_precision": precision,
        "macro_recall": recall,
    }

And lastly set up the TrainingArguments and Trainer. Note that these training parameters could be tuned for better performance.

In [ ]:
from transformers import DataCollatorWithPadding
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./biomedbert_classifier",
    learning_rate=1e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    eval_strategy="epoch",
    save_strategy="no",
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=True,
    report_to="none",
)

data_collator = DataCollatorWithPadding(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

And let's train it. We'll train for just a single epoch, but that should still learn some signal:

In [ ]:
trainer.train()

Lastly, let's get the full performance report on the validation set. The DDI task is typically evaluated using micro-F1 (and only calculated on the positive labels as below).

In [ ]:
from sklearn.metrics import classification_report

predictions = trainer.predict(dataset["validation"])

y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

report = classification_report(
    y_true,
    y_pred,
    labels=[i for i,label in id2label.items() if label != 'NONE'],
    target_names=[label for i,label in id2label.items() if label != 'NONE']
)

print(report)

## Going Further

This classifier idea could be taken further in a few ways:

- Train for more epochs with early stopping (and do hyperparameter tuning)
- Training with all the data may help
- A weighted loss function might help with the class imbalance
- Using the context vectors from the `[Arg1]`/`[Arg2]` tags instead of the [CLS] vector used by the `AutoModelForSequenceClassification` might give a better relation representation